In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import re
import nltk

In [2]:
# تحميل وإعداد البيانات
data = pd.read_csv('../data/NewsCategorizer_modified.csv')
data['text'] = data['headline'] + ' ' + data['short_description'] + ' ' + data['keywords']

# تنظيف البيانات
nltk.download('stopwords')
nltk.download('punkt')
stop_words = set(stopwords.words('english'))


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mahmo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mahmo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
def clean_text(text):
    text = re.sub(r'\W', ' ', text)
    words = word_tokenize(text)
    words = [w for w in words if w.lower() not in stop_words]
    return ' '.join(words)

data['text'] = data['text'].apply(clean_text)

# دمج الفئات
data['category'] = data['category'].replace({
    'FOOD & DRINK': 'LIFESTYLE', 
    'WELLNESS': 'LIFESTYLE',
    'ENTERTAINMENT': 'ENTERTAINMENT',
    'STYLE & BEAUTY': 'ENTERTAINMENT',
    'POLITICS': 'POLITICS',
    'WORLD NEWS': 'POLITICS'
})


In [4]:
# حذف الفئات الغير مرغوب فيها
data = data[~data['category'].isin(['PARENTING', 'TRAVEL'])]

# إعادة ترميز الفئات
label_encoder = LabelEncoder()
data['category'] = label_encoder.fit_transform(data['category'])


In [5]:
# عرض الفئات الجديدة
category_mapping = dict(zip(label_encoder.transform(label_encoder.classes_), label_encoder.classes_))
print(category_mapping)


{0: 'BUSINESS', 1: 'ENTERTAINMENT', 2: 'LIFESTYLE', 3: 'POLITICS', 4: 'SPORTS'}


In [6]:
# تقسيم البيانات
X_train, X_test, y_train, y_test = train_test_split(data['text'], data['category'], test_size=0.2, random_state=42)

# تجهيز النصوص
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(X_train)

X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_test_sequences = tokenizer.texts_to_sequences(X_test)


In [7]:
max_sequence_length = 200
X_train_padded = pad_sequences(X_train_sequences, maxlen=max_sequence_length)
X_test_padded = pad_sequences(X_test_sequences, maxlen=max_sequence_length)

# بناء النموذج
model = Sequential()
model.add(Embedding(input_dim=10000, output_dim=100, input_length=max_sequence_length))
model.add(LSTM(128, return_sequences=True))
model.add(Dropout(0.5))
model.add(LSTM(128))
model.add(Dropout(0.5))
model.add(Dense(len(label_encoder.classes_), activation='softmax'))

# إعداد النموذج
optimizer = Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])


C:\Users\mahmo\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [8]:
# تدريب النموذج
model.fit(X_train_padded, y_train, epochs=15, batch_size=32, validation_data=(X_test_padded, y_test))
    

Epoch 1/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 114s 111ms/step - accuracy: 0.6724 - loss: 0.8455 - val_accuracy: 0.8823 - val_loss: 0.3679
Epoch 2/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 110s 110ms/step - accuracy: 0.9326 - loss: 0.2262 - val_accuracy: 0.8865 - val_loss: 0.3749
Epoch 3/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 111s 111ms/step - accuracy: 0.9578 - loss: 0.1462 - val_accuracy: 0.8845 - val_loss: 0.3912
Epoch 4/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 110s 110ms/step - accuracy: 0.9698 - loss: 0.1079 - val_accuracy: 0.8796 - val_loss: 0.4580
Epoch 5/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 111s 111ms/step - accuracy: 0.9786 - loss: 0.0772 - val_accuracy: 0.8806 - val_loss: 0.5147
Epoch 6/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 127s 127ms/step - accuracy: 0.9829 - loss: 0.0603 - val_accuracy: 0.8802 - val_loss: 0.5188
Epoch 7/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 129s 129ms/step - accuracy: 0.9898 - loss: 0.0396 - val_accuracy: 0.8712 - val_loss: 0.6116
Epoch 8/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 132s 132ms/step - ac

In [9]:
#تقييم النموذج
loss, accuracy = model.evaluate(X_test_padded, y_test)
print(f"Loss: {loss}, Accuracy: {accuracy}")

250/250 ━━━━━━━━━━━━━━━━━━━━ 12s 49ms/step - accuracy: 0.8720 - loss: 0.8288
Loss: 0.793135941028595, Accuracy: 0.874875009059906


In [46]:
# التنبؤ بنص جديد
new_text = "Portuguese striker Cristiano Ronaldo, the Spanish club Real Madrid, joined the ranks of the Italian club Juventus today, Tuesday, in a deal that is the highest in the history of the Italian club, estimated at 105 million euros, according to what the Spanish club Real Madrid announced after the two clubs agreed on the value of the contract, and the deal ended after... Days after a closed meeting that brought together Jorge Mendes, Ronaldo’s agent, Andrea Agnelli, and the president of Juventus, in Greece, after agreeing on the details of the deal."
new_sequence = tokenizer.texts_to_sequences([new_text])
new_padded_sequence = pad_sequences(new_sequence, maxlen=max_sequence_length)


In [47]:
cnn_prediction = model.predict(new_padded_sequence)
cnn_predicted_category_index = np.argmax(cnn_prediction, axis=1)
cnn_predicted_category = label_encoder.inverse_transform(cnn_predicted_category_index)
print(f"CNN Prediction: {cnn_predicted_category[0]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
CNN Prediction: SPORTS
